In [1]:
%%writefile app.py

import streamlit as st
import pandas as pd


st.set_page_config(
    page_title="Khwezi Mining",
    page_icon="💎",
    layout="wide"
)


st.markdown("""
<style>

/* MAIN APP */

.stApp {
    background-color: #F5F6F8;
}


/* HIDE STREAMLIT TOP BAR */

[data-testid="stHeader"] {
    display: none;
}

[data-testid="stToolbar"] {
    display: none;
}

[data-testid="stDecoration"] {
    display: none;
}

#MainMenu {
    visibility: hidden;
}

footer {
    visibility: hidden;
}


/* SIDEBAR */

[data-testid="stSidebar"] {
    background-color: #1B1B1B;
}

[data-testid="stSidebar"] * {
    color: white;
}


/* SIDEBAR NAVIGATION DROPDOWN */

[data-testid="stSidebar"] [data-baseweb="select"] > div {
    background-color: white;
    border-radius: 8px;
}

[data-testid="stSidebar"] [data-baseweb="select"] * {
    color: #1B1B1B !important;
}


/* MAIN HEADINGS */

h1 {
    color: #1B1B1B;
    font-weight: 700;
}

h2, h3 {
    color: #333333;
}


/* METRIC CARDS */

[data-testid="stMetric"] {
    background-color: white;
    padding: 18px;
    border-radius: 12px;
    border-left: 5px solid #D4AF37;
    box-shadow: 0px 2px 8px rgba(0,0,0,0.08);
}


/* BUTTONS */

.stButton > button {
    background-color: #D4AF37;
    color: black;
    border: none;
    border-radius: 8px;
    padding: 8px 20px;
    font-weight: bold;
}

.stButton > button:hover {
    background-color: #C29D2D;
    color: black;
}


/* DOWNLOAD BUTTONS */

.stDownloadButton > button {
    background-color: #1B1B1B;
    color: white;
    border-radius: 8px;
    border: none;
}

.stDownloadButton > button:hover {
    background-color: #D4AF37;
    color: black;
}


/* INPUT BOXES */

.stTextInput input {
    border-radius: 8px;
}

.stNumberInput input {
    border-radius: 8px;
}


/* DATA TABLES */

[data-testid="stDataFrame"] {
    background-color: white;
    border-radius: 10px;
}

</style>
""", unsafe_allow_html=True)


# LOAD CSV FILES

users = pd.read_csv("users.csv")
workers = pd.read_csv("workers.csv")
incidents = pd.read_csv("incidents.csv")
equipment = pd.read_csv("equipment.csv")


# PART A: ACCESS CONTROL

def login(username, password):

    user = users[
        (users["Username"] == username) &
        (users["Password"] == password)
    ]

    if len(user) == 1:
        return user["Role"].item()

    else:
        return "Invalid login"


permissions = {

    "Administrator": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports",
        "Manage Users"
    ],

    "Safety Officer": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Mining Engineer": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Maintenance Engineer": [
        "Dashboard",
        "Equipment",
        "Maintenance",
        "Alerts",
        "Data Analysis",
        "Reports"
    ],

    "Manager": [
        "Dashboard",
        "Worker Safety",
        "Safety Incidents",
        "Equipment",
        "Maintenance",
        "Risk Assessment",
        "Alerts",
        "Data Analysis",
        "Reports"
    ]
}


def get_permissions(role):

    if role in permissions:
        return permissions[role]

    else:
        return []


def show_manage_users():

    global users

    st.title("👤 Manage Users")

    st.subheader("Current Users")

    st.dataframe(
        users[["Username", "Role"]]
    )

    st.subheader("Add New User")

    new_username = st.text_input(
        "New Username"
    )

    new_password = st.text_input(
        "New Password",
        type="password"
    )

    new_role = st.selectbox(
        "New User Role",
        [
            "Administrator",
            "Safety Officer",
            "Mining Engineer",
            "Maintenance Engineer",
            "Manager"
        ],
        key="new_user_role"
    )

    if st.button("Add User"):

        if new_username == "" or new_password == "":

            st.error(
                "Please enter a username and password"
            )

        elif new_username in users["Username"].values:

            st.error(
                "Username already exists"
            )

        else:

            new_user = pd.DataFrame({
                "Username": [new_username],
                "Password": [new_password],
                "Role": [new_role]
            })

            users = pd.concat(
                [users, new_user],
                ignore_index=True
            )

            users.to_csv(
                "users.csv",
                index=False
            )

            st.success(
                "User added successfully"
            )

            st.rerun()


    st.subheader("Change User Role")

    selected_user = st.selectbox(
        "Select User",
        users["Username"],
        key="change_role_user"
    )

    selected_role = st.selectbox(
        "Select New Role",
        [
            "Administrator",
            "Safety Officer",
            "Mining Engineer",
            "Maintenance Engineer",
            "Manager"
        ],
        key="change_role"
    )

    if st.button("Change Role"):

        updated_roles = [""] * len(users)

        for i in range(len(users)):

            if users["Username"][i] == selected_user:

                updated_roles[i] = selected_role

            else:

                updated_roles[i] = users["Role"][i]

        users["Role"] = updated_roles

        users.to_csv(
            "users.csv",
            index=False
        )

        st.success(
            "User role changed successfully"
        )

        st.rerun()


    st.subheader("Delete User")

    delete_user = st.selectbox(
        "Select User to Delete",
        users["Username"],
        key="delete_user"
    )

    if st.button("Delete User"):

        users = users[
            users["Username"] != delete_user
        ].reset_index(drop=True)

        users.to_csv(
            "users.csv",
            index=False
        )

        st.success(
            "User deleted successfully"
        )

        st.rerun()


# PART B: WORKER HEALTH AND SAFETY MONITORING

def worker_risk(
    ppe,
    fatigue,
    near_misses,
    training,
    previous_incidents
):

    score = 0

    if ppe == "No":
        score = score + 2

    if fatigue == "High":
        score = score + 2

    elif fatigue == "Medium":
        score = score + 1

    if near_misses >= 2:
        score = score + 2

    if training == "Incomplete":
        score = score + 2

    if previous_incidents >= 2:
        score = score + 1

    if score >= 6:
        return "Critical"

    elif score >= 4:
        return "High"

    elif score >= 2:
        return "Medium"

    else:
        return "Low"


def safety_warning(
    ppe,
    training,
    fatigue,
    near_misses,
    previous_incidents
):

    warning = ""

    if ppe == "No":
        warning = warning + "PPE non-compliance, "

    if training == "Incomplete":
        warning = warning + "Training incomplete, "

    if fatigue == "High":
        warning = warning + "High fatigue, "

    if near_misses >= 2:
        warning = warning + "Multiple near misses, "

    if previous_incidents >= 2:
        warning = warning + "Multiple previous incidents, "

    if warning == "":
        return "No warning"

    return warning[:-2]


risk_levels = [""] * len(workers)

safety_warnings = [""] * len(workers)


for i in range(len(workers)):

    risk_levels[i] = worker_risk(
        workers["PPE_Compliance"][i],
        workers["Fatigue_Level"][i],
        workers["Near_Misses"][i],
        workers["Training_Status"][i],
        workers["Previous_Incidents"][i]
    )

    safety_warnings[i] = safety_warning(
        workers["PPE_Compliance"][i],
        workers["Training_Status"][i],
        workers["Fatigue_Level"][i],
        workers["Near_Misses"][i],
        workers["Previous_Incidents"][i]
    )


workers["Risk_Level"] = risk_levels

workers["Safety_Warning"] = safety_warnings


def show_worker_safety():

    st.title(
        "👷 Worker Health and Safety"
    )

    risk_filter = st.selectbox(
        "Filter by risk level",
        [
            "All",
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    )

    if risk_filter == "All":

        st.dataframe(
            workers
        )

    else:

        filtered_workers = workers[
            workers["Risk_Level"] == risk_filter
        ]

        st.dataframe(
            filtered_workers
        )

    st.subheader(
        "Workers Requiring Attention"
    )

    attention = workers[
        workers["Safety_Warning"] != "No warning"
    ]

    st.dataframe(
        attention
    )


# PART C: SAFETY INCIDENT DATABASE

def show_safety_incidents():

    global incidents

    st.title(
        "⚠️ Safety Incident Database"
    )

    st.metric(
        "Total Incidents",
        len(incidents)
    )

    search_id = st.text_input(
        "Search Incident ID"
    )

    if search_id != "":

        search_result = incidents[
            incidents["Incident_ID"] == search_id
        ]

        st.dataframe(
            search_result
        )

    severity = st.selectbox(
        "Filter by severity",
        [
            "All",
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    )

    if severity == "All":

        st.dataframe(
            incidents
        )

    else:

        filtered_incidents = incidents[
            incidents["Severity"] == severity
        ]

        st.dataframe(
            filtered_incidents
        )

    st.subheader(
        "Add New Incident"
    )

    with st.form(
        "incident_form"
    ):

        incident_id = st.text_input(
            "Incident ID"
        )

        date = st.date_input(
            "Date"
        )

        time = st.text_input(
            "Time"
        )

        shift = st.selectbox(
            "Shift",
            [
                "Day",
                "Night"
            ]
        )

        location = st.text_input(
            "Location"
        )

        department = st.text_input(
            "Department"
        )

        incident_type = st.text_input(
            "Incident Type"
        )

        incident_severity = st.selectbox(
            "Severity",
            [
                "Low",
                "Medium",
                "High",
                "Critical"
            ]
        )

        injury_status = st.text_input(
            "Injury Status"
        )

        lost_time = st.selectbox(
            "Lost Time Injury",
            [
                "No",
                "Yes"
            ]
        )

        cause = st.text_input(
            "Cause"
        )

        corrective_action = st.text_input(
            "Corrective Action"
        )

        incident_status = st.text_input(
            "Incident Status"
        )

        save_incident = st.form_submit_button(
            "Save Incident"
        )

    if save_incident:

        if incident_id == "":

            st.error(
                "Please enter an Incident ID"
            )

        elif incident_id in incidents[
            "Incident_ID"
        ].values:

            st.error(
                "Incident ID already exists"
            )

        elif location == "":

            st.error(
                "Please enter the incident location"
            )

        elif department == "":

            st.error(
                "Please enter the department"
            )

        elif incident_type == "":

            st.error(
                "Please enter the incident type"
            )

        elif cause == "":

            st.error(
                "Please enter the cause"
            )

        else:

            new_incident = pd.DataFrame({

                "Incident_ID": [incident_id],
                "Date": [str(date)],
                "Time": [time],
                "Shift": [shift],
                "Location": [location],
                "Department": [department],
                "Incident_Type": [incident_type],
                "Severity": [incident_severity],
                "Injury_Status": [injury_status],
                "Lost_Time_Injury": [lost_time],
                "Cause": [cause],
                "Corrective_Action": [corrective_action],
                "Incident_Status": [incident_status]
            })

            incidents = pd.concat(
                [
                    incidents,
                    new_incident
                ],
                ignore_index=True
            )

            incidents.to_csv(
                "incidents.csv",
                index=False
            )

            st.success(
                "Incident saved successfully"
            )

            st.rerun()

    st.subheader(
        "Incident Analysis"
    )

    st.write(
        "Incidents by Severity"
    )

    severity_counts = incidents[
        "Severity"
    ].value_counts()

    st.bar_chart(
        severity_counts
    )

    st.write(
        "Incidents by Department"
    )

    department_counts = incidents[
        "Department"
    ].value_counts()

    st.bar_chart(
        department_counts
    )

    st.write(
        "Incidents by Shift"
    )

    shift_counts = incidents[
        "Shift"
    ].value_counts()

    st.bar_chart(
        shift_counts
    )

    st.write(
        "Incident Types"
    )

    type_counts = incidents[
        "Incident_Type"
    ].value_counts()

    st.bar_chart(
        type_counts
    )

    st.write(
        "Incident Trend"
    )

    incident_trend = incidents[
        "Date"
    ].value_counts().sort_index()

    st.line_chart(
        incident_trend
    )


# PART D: EQUIPMENT DATABASE

def show_equipment_database():

    st.title(
        "🚜 Equipment Monitoring"
    )

    st.subheader(
        "Equipment Database"
    )

    st.dataframe(
        equipment
    )

    st.subheader(
        "Filter by Condition"
    )

    condition = st.selectbox(
        "Select condition",
        [
            "All",
            "Normal",
            "Warning",
            "Critical"
        ]
    )

    if condition == "All":

        st.dataframe(
            equipment
        )

    else:

        filtered_equipment = equipment[
            equipment["Condition_Status"] == condition
        ]

        st.dataframe(
            filtered_equipment
        )


# PART E: EQUIPMENT CONDITION MONITORING

def equipment_condition(
    temperature,
    vibration
):

    if temperature > 100 or vibration > 8:

        return "Critical"

    elif temperature >= 80 or vibration >= 5:

        return "Warning"

    else:

        return "Normal"


condition_status = [""] * len(equipment)


for i in range(len(equipment)):

    condition_status[i] = equipment_condition(
        equipment["Temperature"][i],
        equipment["Vibration"][i]
    )


equipment["Condition_Status"] = condition_status


def show_equipment_condition():

    st.subheader(
        "Equipment Condition Status"
    )

    for i in range(len(equipment)):

        equipment_id = equipment[
            "Equipment_ID"
        ][i]

        status = equipment[
            "Condition_Status"
        ][i]

        if status == "Critical":

            st.error(
                equipment_id +
                " - CRITICAL: Equipment requires inspection"
            )

        elif status == "Warning":

            st.warning(
                equipment_id +
                " - WARNING: Equipment approaching monitoring limits"
            )

        else:

            st.success(
                equipment_id +
                " - NORMAL: Equipment operating within monitoring limits"
            )


# PART F: EQUIPMENT MAINTENANCE MONITORING

availability = [0] * len(equipment)


for i in range(len(equipment)):

    operating_time = equipment[
        "Operating_Hours"
    ][i]

    downtime = equipment[
        "Downtime"
    ][i]

    availability[i] = (
        operating_time /
        (operating_time + downtime)
    ) * 100


equipment["Availability"] = availability


def maintenance_check(
    operating_hours,
    next_maintenance,
    current_status
):

    if current_status == "Under Maintenance":

        return "Under Maintenance"

    elif operating_hours > next_maintenance:

        return "Overdue"

    elif operating_hours >= next_maintenance - 20:

        return "Due Soon"

    else:

        return "Not Due"


maintenance_results = [""] * len(equipment)


for i in range(len(equipment)):

    maintenance_results[i] = maintenance_check(
        equipment["Operating_Hours"][i],
        equipment["Next_Maintenance_Hours"][i],
        equipment["Maintenance_Status"][i]
    )


equipment["Maintenance_Check"] = maintenance_results


def show_maintenance():

    st.title(
        "🛠 Equipment Maintenance"
    )

    st.subheader(
        "Maintenance Status"
    )

    maintenance_attention = equipment[
        equipment["Maintenance_Check"] != "Not Due"
    ]

    st.dataframe(
        maintenance_attention
    )

    st.subheader(
        "Low Availability"
    )

    low_availability_equipment = equipment[
        equipment["Availability"] < 85
    ]

    st.dataframe(
        low_availability_equipment
    )

    st.subheader(
        "Maintenance Alerts"
    )

    for i in range(len(equipment)):

        equipment_id = equipment[
            "Equipment_ID"
        ][i]

        if equipment[
            "Maintenance_Check"
        ][i] == "Overdue":

            st.error(
                equipment_id +
                " - Maintenance overdue"
            )

        elif equipment[
            "Maintenance_Check"
        ][i] == "Due Soon":

            st.warning(
                equipment_id +
                " - Maintenance due soon"
            )

        if equipment[
            "Maintenance_Status"
        ][i] == "Under Maintenance":

            st.info(
                equipment_id +
                " - Currently under maintenance"
            )

        if equipment[
            "Downtime"
        ][i] > 30:

            st.warning(
                equipment_id +
                " - Excessive downtime"
            )

        if equipment[
            "Previous_Alerts"
        ][i] >= 3:

            st.warning(
                equipment_id +
                " - Repeated equipment alerts"
            )

        if equipment[
            "Availability"
        ][i] < 85:

            st.warning(
                equipment_id +
                " - Low equipment availability"
            )


# PART G: RISK ASSESSMENT

def risk_assessment(
    likelihood,
    consequence
):

    score = likelihood * consequence

    if score <= 4:

        level = "Low"

    elif score <= 9:

        level = "Medium"

    elif score <= 16:

        level = "High"

    else:

        level = "Critical"

    return score, level


def show_risk_assessment():

    st.title(
        "📋 Risk Assessment"
    )

    likelihood = st.number_input(
        "Likelihood",
        min_value=1,
        max_value=5,
        value=1
    )

    consequence = st.number_input(
        "Consequence",
        min_value=1,
        max_value=5,
        value=1
    )

    if st.button(
        "Calculate Risk"
    ):

        score, level = risk_assessment(
            likelihood,
            consequence
        )

        st.write(
            "Risk Score:",
            score
        )

        st.write(
            "Risk Level:",
            level
        )

        if level == "Critical":

            st.error(
                "CRITICAL RISK"
            )

        elif level == "High":

            st.warning(
                "HIGH RISK"
            )

        else:

            st.success(
                "Risk assessment completed"
            )


# PART H: ALERTS AND NOTIFICATIONS

def equipment_alert(
    temperature,
    vibration,
    brakes,
    tyres,
    engine,
    maintenance
):

    alert = ""

    if temperature > 100:

        alert = alert + "Critical temperature, "

    elif temperature >= 80:

        alert = alert + "High temperature, "

    if vibration > 8:

        alert = alert + "Critical vibration, "

    elif vibration >= 5:

        alert = alert + "High vibration, "

    if brakes != "Good":

        alert = alert + "Check brakes, "

    if tyres != "Good":

        alert = alert + "Check tyres, "

    if engine != "Good":

        alert = alert + "Check engine, "

    if maintenance == "Overdue":

        alert = alert + "Maintenance overdue, "

    if alert == "":

        return "No alert"

    return alert[:-2]


equipment_alerts = [""] * len(equipment)


for i in range(len(equipment)):

    equipment_alerts[i] = equipment_alert(
        equipment["Temperature"][i],
        equipment["Vibration"][i],
        equipment["Brake_Status"][i],
        equipment["Tyre_Status"][i],
        equipment["Engine_Status"][i],
        equipment["Maintenance_Check"][i]
    )


equipment["Equipment_Alert"] = equipment_alerts


def show_alerts(role):

    st.title(
        "🚨 Alerts and Notifications"
    )

    if role != "Maintenance Engineer":

        st.subheader(
            "Worker Safety Alerts"
        )

        worker_alerts = workers[
            workers["Safety_Warning"] != "No warning"
        ]

        st.dataframe(
            worker_alerts
        )

        for i in range(len(workers)):

            if workers[
                "Risk_Level"
            ][i] == "Critical":

                st.error(
                    "Critical worker safety alert: "
                    + workers["Worker_ID"][i]
                )

            elif workers[
                "Risk_Level"
            ][i] == "High":

                st.warning(
                    "High worker safety risk: "
                    + workers["Worker_ID"][i]
                )

    if role != "Safety Officer":

        st.subheader(
            "Equipment Alerts"
        )

        equipment_warning_table = equipment[
            equipment["Equipment_Alert"] != "No alert"
        ]

        st.dataframe(
            equipment_warning_table
        )

        for i in range(len(equipment)):

            if equipment[
                "Condition_Status"
            ][i] == "Critical":

                st.error(
                    "Critical equipment alert: "
                    + equipment["Equipment_ID"][i]
                )

            elif equipment[
                "Condition_Status"
            ][i] == "Warning":

                st.warning(
                    "Equipment warning: "
                    + equipment["Equipment_ID"][i]
                )


# PART I: MINING MONITORING DASHBOARD

def show_dashboard():

    st.title(
        "📊 Mining Monitoring Dashboard"
    )

    total_workers = len(workers)

    total_incidents = len(incidents)

    total_near_misses = workers[
        "Near_Misses"
    ].sum()

    total_equipment = len(equipment)

    ppe_yes = 0

    high_risk_observations = 0

    for i in range(len(workers)):

        if workers[
            "PPE_Compliance"
        ][i] == "Yes":

            ppe_yes = ppe_yes + 1

        if workers[
            "Risk_Level"
        ][i] == "High":

            high_risk_observations = (
                high_risk_observations + 1
            )

        elif workers[
            "Risk_Level"
        ][i] == "Critical":

            high_risk_observations = (
                high_risk_observations + 1
            )

    if len(workers) > 0:

        ppe_percentage = (
            ppe_yes /
            len(workers)
        ) * 100

    else:

        ppe_percentage = 0

    available = 0

    under_maintenance = 0

    equipment_alert_count = 0

    critical_alerts = 0

    for i in range(len(equipment)):

        if equipment[
            "Maintenance_Status"
        ][i] == "Available":

            available = available + 1

        if equipment[
            "Maintenance_Status"
        ][i] == "Under Maintenance":

            under_maintenance = (
                under_maintenance + 1
            )

        if equipment[
            "Equipment_Alert"
        ][i] != "No alert":

            equipment_alert_count = (
                equipment_alert_count + 1
            )

        if equipment[
            "Condition_Status"
        ][i] == "Critical":

            critical_alerts = (
                critical_alerts + 1
            )

    if len(equipment) > 0:

        average_availability = (
            equipment["Availability"].sum() /
            len(equipment)
        )

    else:

        average_availability = 0

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Total Workers",
        total_workers
    )

    col2.metric(
        "Safety Incidents",
        total_incidents
    )

    col3.metric(
        "Near Misses",
        total_near_misses
    )

    col4.metric(
        "High-Risk Observations",
        high_risk_observations
    )

    col5, col6, col7, col8 = st.columns(4)

    col5.metric(
        "PPE Compliance",
        str(round(ppe_percentage, 1)) + "%"
    )

    col6.metric(
        "Total Equipment",
        total_equipment
    )

    col7.metric(
        "Equipment Available",
        available
    )

    col8.metric(
        "Under Maintenance",
        under_maintenance
    )

    col9, col10, col11 = st.columns(3)

    col9.metric(
        "Equipment Alerts",
        equipment_alert_count
    )

    col10.metric(
        "Average Availability",
        str(round(average_availability, 1)) + "%"
    )

    col11.metric(
        "Critical Equipment Alerts",
        critical_alerts
    )


# PART J: INTERACTIVE DATA ANALYSIS

def most_common(column):

    counts = {}

    for i in range(len(column)):

        item = column[i]

        if item in counts:

            counts[item] = counts[item] + 1

        else:

            counts[item] = 1

    answer = ""

    highest = 0

    for item in counts:

        if counts[item] > highest:

            highest = counts[item]

            answer = item

    return answer


def highest_downtime():

    equipment_id = equipment[
        "Equipment_ID"
    ][0]

    highest = equipment[
        "Downtime"
    ][0]

    for i in range(len(equipment)):

        if equipment[
            "Downtime"
        ][i] > highest:

            highest = equipment[
                "Downtime"
            ][i]

            equipment_id = equipment[
                "Equipment_ID"
            ][i]

    return equipment_id


def lowest_availability():

    equipment_id = equipment[
        "Equipment_ID"
    ][0]

    lowest = equipment[
        "Availability"
    ][0]

    for i in range(len(equipment)):

        if equipment[
            "Availability"
        ][i] < lowest:

            lowest = equipment[
                "Availability"
            ][i]

            equipment_id = equipment[
                "Equipment_ID"
            ][i]

    return equipment_id


def serious_incident_count():

    count = 0

    for i in range(len(incidents)):

        if incidents[
            "Severity"
        ][i] == "High":

            count = count + 1

        elif incidents[
            "Severity"
        ][i] == "Critical":

            count = count + 1

    return count


def ppe_compliance_percentage():

    compliant = 0

    for i in range(len(workers)):

        if workers[
            "PPE_Compliance"
        ][i] == "Yes":

            compliant = compliant + 1

    if len(workers) > 0:

        percentage = (
            compliant /
            len(workers)
        ) * 100

    else:

        percentage = 0

    return round(
        percentage,
        1
    )


def highest_temperature():

    equipment_id = equipment[
        "Equipment_ID"
    ][0]

    highest = equipment[
        "Temperature"
    ][0]

    for i in range(len(equipment)):

        if equipment[
            "Temperature"
        ][i] > highest:

            highest = equipment[
                "Temperature"
            ][i]

            equipment_id = equipment[
                "Equipment_ID"
            ][i]

    return equipment_id


def highest_vibration():

    equipment_id = equipment[
        "Equipment_ID"
    ][0]

    highest = equipment[
        "Vibration"
    ][0]

    for i in range(len(equipment)):

        if equipment[
            "Vibration"
        ][i] > highest:

            highest = equipment[
                "Vibration"
            ][i]

            equipment_id = equipment[
                "Equipment_ID"
            ][i]

    return equipment_id


questions = [

    "Which department has the most incidents?",

    "Which shift has the most incidents?",

    "What is the most common incident type?",

    "What is the most common cause?",

    "Which equipment has the highest downtime?",

    "Which equipment has the lowest availability?",

    "How many incidents are high or critical?",

    "What percentage of workers comply with PPE requirements?",

    "Which equipment has the highest temperature?",

    "Which equipment has the highest vibration?"
]


def answer_question(question):

    if question == "Which department has the most incidents?":

        return most_common(
            incidents["Department"]
        )

    elif question == "Which shift has the most incidents?":

        return most_common(
            incidents["Shift"]
        )

    elif question == "What is the most common incident type?":

        return most_common(
            incidents["Incident_Type"]
        )

    elif question == "What is the most common cause?":

        return most_common(
            incidents["Cause"]
        )

    elif question == "Which equipment has the highest downtime?":

        return highest_downtime()

    elif question == "Which equipment has the lowest availability?":

        return lowest_availability()

    elif question == "How many incidents are high or critical?":

        return serious_incident_count()

    elif question == "What percentage of workers comply with PPE requirements?":

        return str(
            ppe_compliance_percentage()
        ) + "%"

    elif question == "Which equipment has the highest temperature?":

        return highest_temperature()

    elif question == "Which equipment has the highest vibration?":

        return highest_vibration()

    else:

        return "Question not available"


def show_data_analysis(role):

    st.title(
        "📈 Interactive Data Analysis"
    )

    safety_questions = [

        "Which department has the most incidents?",

        "Which shift has the most incidents?",

        "What is the most common incident type?",

        "What is the most common cause?",

        "How many incidents are high or critical?",

        "What percentage of workers comply with PPE requirements?"
    ]

    equipment_questions = [

        "Which equipment has the highest downtime?",

        "Which equipment has the lowest availability?",

        "Which equipment has the highest temperature?",

        "Which equipment has the highest vibration?"
    ]

    if role == "Safety Officer":

        allowed_questions = safety_questions

    elif role == "Maintenance Engineer":

        allowed_questions = equipment_questions

    else:

        allowed_questions = questions

    question = st.selectbox(
        "Choose a mining question",
        allowed_questions
    )

    if st.button(
        "Answer Question"
    ):

        answer = answer_question(
            question
        )

        st.success(
            "Answer: " +
            str(answer)
        )

    if role != "Maintenance Engineer":

        st.subheader(
            "Safety Analysis"
        )

        st.write(
            "Incidents by Department"
        )

        st.bar_chart(
            incidents[
                "Department"
            ].value_counts()
        )

        st.write(
            "Incidents by Severity"
        )

        st.bar_chart(
            incidents[
                "Severity"
            ].value_counts()
        )

    if role != "Safety Officer":

        st.subheader(
            "Equipment Analysis"
        )

        st.write(
            "Equipment Condition"
        )

        st.bar_chart(
            equipment[
                "Condition_Status"
            ].value_counts()
        )


# REPORTS

def show_reports(role):

    st.title(
        "📄 Reports"
    )

    if role != "Maintenance Engineer":

        st.subheader(
            "Worker Safety Report"
        )

        st.dataframe(
            workers
        )

        worker_csv = workers.to_csv(
            index=False
        )

        st.download_button(
            "Download Worker Report",
            worker_csv,
            "worker_report.csv"
        )

        st.subheader(
            "Safety Incident Report"
        )

        st.dataframe(
            incidents
        )

        incident_csv = incidents.to_csv(
            index=False
        )

        st.download_button(
            "Download Incident Report",
            incident_csv,
            "incident_report.csv"
        )

    if role != "Safety Officer":

        st.subheader(
            "Equipment Report"
        )

        st.dataframe(
            equipment
        )

        equipment_csv = equipment.to_csv(
            index=False
        )

        st.download_button(
            "Download Equipment Report",
            equipment_csv,
            "equipment_report.csv"
        )


# LOGIN SESSION

if "logged_in" not in st.session_state:

    st.session_state.logged_in = False


if "role" not in st.session_state:

    st.session_state.role = None


# LOGIN PAGE

if st.session_state.logged_in == False or st.session_state.role == None:

    st.markdown(
        """
        <style>

        .block-container {
            padding-top: 0.3rem !important;
            padding-bottom: 1rem !important;
        }

        div[data-testid="stImage"] {
            margin-bottom: -1rem;
        }

        </style>
        """,
        unsafe_allow_html=True
    )


    left_logo, logo_column, right_logo = st.columns(
        [1.2, 1.5, 1.2]
    )


    with logo_column:

        st.image(
            "Khwezi Mining Diamond Emblem.png",
            width="stretch"
        )


    # MONITORING APPLICATION NAME

    st.markdown(
        """
        <div style="
            text-align: center;
            color: #1B1B1B;
            font-size: 20px;
            font-weight: 600;
            margin-top: -6px;
            margin-bottom: 20px;
        ">
            Mining Health, Safety & Equipment Monitoring Application
        </div>
        """,
        unsafe_allow_html=True
    )


    left, middle, right = st.columns(
        [1.4, 1.3, 1.4]
    )


    with middle:

        username = st.text_input(
            "Username"
        )

        password = st.text_input(
            "Password",
            type="password"
        )

        login_button = st.button(
            "Login",
            width="stretch"
        )


    if login_button:

        role = login(
            username,
            password
        )

        if role == "Invalid login":

            st.error(
                "Invalid username or password"
            )

        else:

            st.session_state.logged_in = True

            st.session_state.role = role

            st.rerun()


# MAIN APPLICATION

else:

    role = st.session_state.role


    # SIDEBAR BRAND

    st.sidebar.markdown(
        '<div style="display:flex; align-items:center; gap:12px; margin-bottom:20px;">'
        '<span style="color:#D4AF37; font-size:30px;">◆</span>'
        '<span style="color:white; font-size:24px; font-weight:700;">Khwezi Mining</span>'
        '</div>',
        unsafe_allow_html=True
    )


    st.sidebar.write(
        "Role:",
        role
    )


    pages = get_permissions(
        role
    )


    page = st.sidebar.selectbox(
        "Navigation",
        pages
    )


    if st.sidebar.button(
        "Logout"
    ):

        st.session_state.logged_in = False

        st.session_state.role = None

        st.rerun()


    # PART A

    if page == "Manage Users":

        show_manage_users()


    # PART B

    elif page == "Worker Safety":

        show_worker_safety()


    # PART C

    elif page == "Safety Incidents":

        show_safety_incidents()


    # PART D AND PART E

    elif page == "Equipment":

        show_equipment_database()

        show_equipment_condition()


    # PART F

    elif page == "Maintenance":

        show_maintenance()


    # PART G

    elif page == "Risk Assessment":

        show_risk_assessment()


    # PART H

    elif page == "Alerts":

        show_alerts(
            role
        )


    # PART I

    elif page == "Dashboard":

        show_dashboard()


    # PART J

    elif page == "Data Analysis":

        show_data_analysis(
            role
        )


    # REPORTS

    elif page == "Reports":

        show_reports(
            role
        )

Overwriting app.py


In [ ]:
!streamlit run app.py